In [12]:
import requests
import pandas as pd
import numpy as np
import os
import sqlite3 as sql
import matplotlib.pyplot as plt
import time

apiKey = "RGAPI-d54fbc39-37bf-4af8-b517-9903d66e77be"
gameName = 'Shadow'
tagLine = 'MC3'
puuid = '4RWZ8j41vOINfON_3oxDLVn4rgUSRYU6UTwRg-YVxWejFdHIFvQb9j-7gL4tGCLzlMgLjYZL4LAbTA'

#gets unique ID number for a player from their visable in-game name+tagline or summoner name
def get_puuid(summonerId=None, gameName = None, tagLine=None):
    if summonerId is not None:
        link = 'https://na1.api.riotgames.com/lol/summoner/v4/summoners/'
        response = requests.get(link + summonerId + "?api_key=" + apiKey)
        return response.json()['puuid']
    else:
        link = f'https://americas.api.riotgames.com/riot/account/v1/accounts/by-riot-id/{gameName}/{tagLine}?api_key={apiKey}'
        response = requests.get(link)
        return response.json()['puuid']

In [13]:
#gets the in-game name+tagline given a puuid
def get_idtag(puuid=None):
    link = 'https://americas.api.riotgames.com/riot/account/v1/accounts/by-puuid'
    response = requests.get(link + puuid + "?api_key=" + apiKey)
    id = {'gameName': response.json()['gameName'], 'tagLine': response.json()['tagLine']}
    return id

In [ ]:
#Gets the top players (base top 300 players max unless specified) in all of League of Legends. Top ranks are Master->Grandmaster->Challenger w/ Challenger being the top rank
def get_ladder(top=300):
    root = 'https://na1.api.riotgames.com'
    chall = '/lol/league/v4/challengerleagues/by-queue/RANKED_SOLO_5x5'
    gm = '/lol/league/v4/grandmasterleagues/by-queue/RANKED_SOLO_5x5'
    master = '/lol/league/v4/masterleagues/by-queue/RANKED_SOLO_5x5'

    chall_response = requests.get(root + chall + "?api_key=" + apiKey)
    gm_response = requests.get(root + gm + "?api_key=" + apiKey)
    master_response = requests.get(root + master + "?api_key=" + apiKey)

    chall_df = pd.DataFrame(chall_response.json()['entries'])
    gm_df = pd.DataFrame(gm_response.json()['entries'])
    master_df = pd.DataFrame(master_response.json()['entries'])

    solo_queue_df = pd.concat([chall_df, gm_df, master_df]).reset_index(drop=True)
    solo_queue_df = solo_queue_df.sort_values(by='leaguePoints', ascending=False)
    solo_queue_df = solo_queue_df.head(top)
    solo_queue_df = solo_queue_df.drop(columns= ['rank', 'inactive', 'freshBlood'], errors='ignore')
    return solo_queue_df.reset_index(drop=True)

In [15]:
#Gets the lastest (default latest 20) games played by a player from their puuid
def get_match_history(puuid=None, start=0, count=20):
    link = f'https://americas.api.riotgames.com/lol/match/v5/matches/by-puuid/{puuid}/ids?'
    params = f'start={start}&count={count}'

    response = requests.get(link + params + '&api_key=' + apiKey)
    if response.status_code == 200:
        return response.json()
    else: return []

#Returns the json for a specific match from a matchId number from someone's match history
def get_match(matchId=None):
    link = f'https://americas.api.riotgames.com'
    param = f'/lol/match/v5/matches/{matchId}'

    response = requests.get(link + param + '?api_key=' + apiKey)
    return response.json()

In [16]:
#Processes the data from a match into multiple dataframes for the match metadata, team-wide data, and the multitude of stats for all players for the database
def process_match(match_json):
    metadata = match_json['metadata']
    info = match_json['info']
    matchId = match_json['metadata']['matchId']

    game_start = info['gameStartTimestamp']
    game_end = info.get('gameEndTimestamp', game_start + (info['gameDuration'] * 1000))

    match_df = pd.DataFrame([{
        #basic match metadata
        'match_id': matchId,
        'game_version': info['gameVersion'],
        'game_duration': info['gameDuration'],
        'time_played_sec': (game_end - game_start) / 1000,
        'queue_id': info['queueId'],
        'game_mode': info['gameMode']
    }])

    teams = match_json['info']['teams']
    team_records = []
    for team in teams:
        #team-wide data for each match, mainly objectives
        obj = team['objectives']
        ban_ids = ",".join([str(ban['championId']) for ban in team.get('bans', [])])

        team_records.append({
            'match_id': matchId,
            'team_id': team['teamId'],
            'win': 1 if team['win'] else 0,
            'bans': ban_ids,

            'baron_kills': obj['baron']['kills'],
            'baron_first': 1 if obj['baron']['first'] else 0,
            
            'champion_kills': obj['champion']['kills'],
            'champion_first': 1 if obj['champion']['first'] else 0,

            'dragon_kills': obj['dragon']['kills'],
            'dragon_first': 1 if obj['dragon']['first'] else 0,

            'grubs_kills': obj['horde']['kills'],
            'grubs_first': 1 if obj['horde']['first'] else 0,

            'inhibitor_kills': obj['inhibitor']['kills'],
            'inhibitor_first': 1 if obj['inhibitor']['first'] else 0,

            'tower_kills': obj['tower']['kills'],
            'tower_first': 1 if obj['tower']['first'] else 0
        })
    team_df = pd.DataFrame(team_records)

    participants = match_json['info']['participants']
    players_list = []
    
    for player in participants:
        #common player data for each match
        player_data = {
            'match_id': matchId,
            'puuid': player['puuid'],
            'riotIdGameName': player['riotIdGameName'],
            'riotIdTagline': player['riotIdTagline'],
            'team_id': player['teamId'],
            'win': 1 if player['win'] else 0,
            'gameEndedInEarlySurrender': 1 if player['gameEndedInEarlySurrender'] else 0,
            'gameEndedInSurrender': 1 if player['gameEndedInSurrender'] else 0,

            'championId': player['championId'],
            'championName': player['championName'],
            'teamPosition': player['teamPosition'],

            'kills': player['kills'],
            'deaths': player['deaths'],
            'assists': player['assists'],
            'summOne': player['summoner1Id'],
            'summTwo': player['summoner2Id'],
            'totalMinionsKilled': player['totalMinionsKilled'],

            'largestKillingSpree': player['largestKillingSpree'],
            'longestTimeSpentLiving': player['longestTimeSpentLiving'],
            'totalTimeSpentDead': player['totalTimeSpentDead'],

            'goldEarned': player['goldEarned'],
            'goldSpent': player['goldSpent'],

            'item0': player['item0'],
            'item1': player['item1'],
            'item2': player['item2'],
            'item3': player['item3'],
            'item4': player['item4'],
            'item5': player['item5'],
            'item6': player['item6'],

            'firstBloodKill': 1 if player['firstBloodKill'] else 0,
            'firstBloodAssist': 1 if player['firstBloodAssist'] else 0,
            'firstTowerKill': 1 if player['firstTowerKill'] else 0,
            'firstTowerAssist': 1 if player['firstTowerAssist'] else 0,
            'turretTakedowns': player['turretTakedowns'],
            'dragonKills': player['dragonKills'],

            'damageDealtToBuildings': player['damageDealtToBuildings'],
            'damageDealtToObjectives': player['damageDealtToObjectives'],
            'damageDealtToTurrets': player['damageDealtToTurrets'],

            'totalDamageDealtToChampions': player['totalDamageDealtToChampions'],
            'totalDamageTaken': player['totalDamageTaken'],
            'totalDamageShieldedOnTeammates': player['totalDamageShieldedOnTeammates'],
            'totalHeal': player['totalHeal'],
            'totalHealsOnTeammates': player['totalHealsOnTeammates'],
            'totalDamageTaken': player['totalDamageTaken'],
            'totalTimeCCDealt': player['totalTimeCCDealt'],

            'totalAllyJungleMinionsKilled': player['totalAllyJungleMinionsKilled'],
            'totalEnemyJungleMinionsKilled': player['totalEnemyJungleMinionsKilled'],

            'visionScore': player['visionScore'],
            'wardsPlaced': player['wardsPlaced'],
            'wardsKilled': player['wardsKilled'],
            'visionWardsBoughtInGame': player['visionWardsBoughtInGame'],
        }
        #extra unique challenges that may see more specific correlation with winning, using get() here as some challenges might not exist in some gamemodes and might error otherwise
        target_challenges = ['goldPerMinute', 'bountyGold', 'killParticipation', 'turretPlatesTaken', 'dragonTakedowns', 'soloKills', 'takedownsFirstXMinutes', 'takedownsFirst25Minutes', 
                             'damagePerMinute', 'earlyLaningPhaseGoldExpAdvantage', 'laneMinionsFirst10Minutes', 'laningPhaseGoldExpAdvantage', 'maxCsAdvantageOnLaneOpponent', 
                             'maxLevelLeadLaneOpponent', 'killsOnOtherLanesEarlyJungleAsLaner', 'fastestLegendary', 'enemyChampionImmobilizations', 'maxKillDeficit', 'outnumberedKills', 
                            'pickKillWithAlly', 'saveAllyFromDeath', 'soloTurretsLategame', 'junglerKillsEarlyJungle', 'killsOnLanersEarlyJungleAsJungler', 'scuttleCrabKills', 
                            'jungleCsBefore10Minutes', 'enemyJungleMonsterKills', 'junglerTakedownsNearDamagedEpicMonster', 'visionScoreAdvantageLaneOpponent', 'visionScorePerMinute', 
                            'controlWardsPlaced', 'controlWardTimeCoverageInRiverOrEnemyHalf']
        challenges = player.get('challenges', {})
        for key in target_challenges:
            player_data[key] = challenges.get(key, 0)

        players_list.append(player_data)

    players_df = pd.DataFrame(players_list)

    return match_df, team_df, players_df

In [17]:
#Returns a json of the entire match timeline for a given match id
def get_match_timeline_json(matchId):
    link = f'https://americas.api.riotgames.com/lol/match/v5/matches/{matchId}/timeline'
    response = requests.get(link + '?api_key=' + apiKey)
    return response.json()

#Returns players stats from a match timeline at the given timestamp
def extract_timeline_stats(timeline_json, timestamp=15):
    timeframe = timeline_json['info']['frames'][timestamp]
    player_data = timeframe['participantFrames']

    stats_at_time = []
    for playerId, data in player_data.items():
        stats_at_time.append({'participantId': playerId, 'totalGold': data['totalGold'], 'xp': data['xp'], 'minionsKilled': data['minionsKilled'] + data['jungleMonstersKilled']})

    return stats_at_time

#Adds columns for gold, xp, minions killed, and gold difference vs lane opponent to player data frames at given timestamps. Can insert multiple timestamps at once
def match_with_timeline(match_json, timeline_json, timestamps = [15]):
    match_df, team_df, players_df = process_match(match_json)

    for minute in timestamps:
        if len(timeline_json['info']['frames']) <= minute:
            continue

        stats_list = extract_timeline_stats(timeline_json, minute)
        snapshot_at_tf = {}

        for participant in stats_list:
            playerId = int(participant['participantId'])
            snapshot_at_tf[playerId] = participant

        gold_ts, xp_ts, minions_ts, gold_diff_ts = [], [], [], []

        for i in range(1, 11):
            player_stats = snapshot_at_tf.get(i, {'totalGold': 0, 'xp': 0, 'minionsKilled': 0})

            opponent_id = i + 5 if i <= 5 else i - 5
            opponent_stats = snapshot_at_tf.get(opponent_id,{'totalGold': 0, 'xp': 0, 'minionsKilled': 0})

            gold_ts.append(player_stats['totalGold'])
            xp_ts.append(player_stats['xp'])
            minions_ts.append(player_stats['minionsKilled'])
            gold_diff_ts.append(player_stats['gold'] - opponent_stats['gold'])

        players_df[f'goldAt{minute}'] = gold_ts
        players_df[f'xpAt{minute}'] = xp_ts
        players_df[f'minionsAt{minute}'] = minions_ts
        players_df[f'goldDiffAt{minute}'] = gold_diff_ts

    return match_df, team_df, players_df

In [18]:
#Checks if a match id has already been processed to avoid duplicates. Makes sure the database exists first
def is_match_processed(matchId, db_path='league_data.db'):
    if not os.path.exists(db_path):
        return False

    conn = sql.connect(db_path)
    cursor = conn.cursor()
    query = f"SELECT 1 FROM matches WHERE match_id = '{matchId}'"
    
    cursor.execute(query)
    result = cursor.fetchone()
    conn.close()

    return result is not None


#Saves the dataframes returned by process_match() to the database
def save_to_db(match_df, team_df, players_df, db_path='league_data.db'):
    conn = sql.connect(db_path)

    try: 
        match_df.to_sql('matches', conn, if_exists='append', index=False)
        team_df.to_sql('teams', conn, if_exists='append', index=False)
        players_df.to_sql('participants', conn, if_exists='append', index=False)
        conn.commit()
        print(f"Saved match: {match_df['match_id'].iloc[0]}")

    except Exception as e:

        actual_error = e.__cause__ if e.__cause__ else e
        print(f"Database Error: {actual_error}")
        conn.rollback()

    finally: conn.close()

In [19]:
#Returns the processed match data for all matches from a specific player's puuid
def get_av_matches(puuid=None):
    matchIds = get_match_history(puuid=puuid)
    all_matches = []

    for matchId in matchIds:
        game = get_match(matchId=matchId)
        match_df = process_match(game, puuid=puuid)
        all_matches.append(match_df)

    av_df = pd.concat(all_matches, ignore_index=True)
    return av_df

#Returns a unique list of matches from the top ranked players returned from get_ladder(). Gets the last (default 20) matches for each top ranked players before filtering out only unique matchIDs
def get_top_matches(ladder, count=20):
    all_matches = []

    for index, player in ladder.iterrows():
        player_puuid = player.get('puuid')
        if not player_puuid:
            continue

        print(f'Fetching match history for player {index + 1}...')

        player_matches = get_match_history(puuid=player_puuid, count=count)
        
        all_matches.extend(player_matches)
        time.sleep(1.2)

    unique_top_matches = list(set(all_matches))
    top_matches_df = pd.DataFrame(unique_top_matches, columns=['matchId'])
    return top_matches_df


In [ ]:
#Start of main function, testing, cell shows the top of the solo queue ladder from the top ranked players
ladder_df = get_ladder(top=5)
ladder_df

,puuid,leaguePoints,wins,losses,veteran,hotStreak
0,8Lm1tpEXpao632CUo_0rmJRbPIxiMe0befZ6VipGE1eveO...,924,141,37,False,False
1,AJjsIjqSxucdYr_LjdfHDmSS4US51CdsD3E9tHz42C-jL6...,855,104,24,False,False
2,VF4fY-U7JCtU0IRAGQGdpgeurU8t1aRbWwK1eXbAKnJgPZ...,829,329,278,False,False
3,Dv6PoH2g7P4vbDxLluzncPneLmJ2jjxSU27dKswua0whbg...,826,174,177,False,True
4,44dBsY50V05YIQZUP_TFyWu16H_s9HSUNfxCQA4qmxC4Fu...,824,148,65,False,True


In [ ]:
#Fetch match history of match ids for COUNT games for every top player
match_ids = get_top_matches(ladder_df, count=5)
match_ids

Fetching match history for player 1...
Fetching match history for player 2...
Fetching match history for player 3...
Fetching match history for player 4...
Fetching match history for player 5...


,matchId
0,NA1_5551012965
1,NA1_5552050217
2,NA1_5550930831
3,NA1_5551888262
4,NA1_5552010109
5,NA1_5552055643
6,NA1_5551838818
7,NA1_5552030225
8,NA1_5551826685
9,NA1_5552083944


In [25]:
#Processes matches and saves them to league_data.db. Displays The 3 tables matches, teams, participants at the end
for match_id in match_ids['matchId']:
    if is_match_processed(match_id):
        continue
    try:
        match_data = get_match(match_id)
        match_df, team_df, players_df = process_match(match_data)

        save_to_db(match_df, team_df, players_df)
    except Exception as e:
        print(f"Error processing match {match_id}: {e}")

    time.sleep(1.2)
print("Successfully saved all match data to database")

conn = sql.connect('league_data.db')

df_matches = pd.read_sql("SELECT * FROM matches LIMIT 5", conn)
df_teams = pd.read_sql("SELECT * FROM teams LIMIT 5", conn)
df_participants = pd.read_sql("SELECT * FROM participants LIMIT 5", conn)

conn.close()

display(df_matches, df_teams, df_participants)

Successfully saved all match data to database


,match_id,game_version,game_duration,time_played_sec,queue_id,game_mode
0,NA1_5551012965,16.9.771.8383,1607,1607.255,420,CLASSIC
1,NA1_5552050217,16.9.772.1032,1360,1358.452,420,CLASSIC
2,NA1_5550930831,16.9.771.8383,1445,1444.720,420,CLASSIC
3,NA1_5551888262,16.9.772.1032,1571,1570.152,420,CLASSIC
4,NA1_5552010109,16.9.772.1032,1905,1904.900,420,CLASSIC


,match_id,team_id,win,bans,baron_kills,baron_first,champion_kills,champion_first,dragon_kills,dragon_first,grubs_kills,grubs_first,inhibitor_kills,inhibitor_first,tower_kills,tower_first
0,NA1_5551012965,100,0,"104,107,238,119,85",0,0,16,1,0,0,3,1,0,0,2,0
1,NA1_5551012965,200,1,"10,200,799,201,157",1,1,29,0,4,1,0,0,1,1,9,1
2,NA1_5552050217,100,1,"266,81,64,61,555",0,0,21,1,2,0,3,1,1,1,9,1
3,NA1_5552050217,200,0,"104,950,910,147,157",0,0,16,0,1,1,0,0,0,0,1,0
4,NA1_5550930831,100,1,"85,-1,555,119,7",1,1,34,1,3,1,3,1,2,1,11,1


,match_id,puuid,riotIdGameName,riotIdTagline,team_id,win,gameEndedInEarlySurrender,gameEndedInSurrender,championId,championName,...,junglerKillsEarlyJungle,killsOnLanersEarlyJungleAsJungler,scuttleCrabKills,jungleCsBefore10Minutes,enemyJungleMonsterKills,junglerTakedownsNearDamagedEpicMonster,visionScoreAdvantageLaneOpponent,visionScorePerMinute,controlWardsPlaced,controlWardTimeCoverageInRiverOrEnemyHalf
0,NA1_5551012965,i003rczmtm5sBrMQRtwZNQ20t7FpCJ4hSqffDIj3w62qzc...,King Kazi,WKND,100,0,0,0,897,KSante,...,0,0,0,0.0,0,0,0.712226,0.832203,0,0.000000
1,NA1_5551012965,mrq83AnaREhDPqDorCObO2t3965TU_wldoiO6_g16pfsbC...,The Finest One,UwU,100,0,0,0,63,Brand,...,1,0,3,70.0,6,0,-0.064859,0.933746,1,0.000000
2,NA1_5551012965,kfypHgc2XvhWJYXfuDduVjndVLi6WfkyUKwDlRcVt5JZpY...,Truand Déglingué,Fent,100,0,0,0,134,Syndra,...,0,0,0,0.0,0,0,-0.040233,0.528460,0,0.000000
3,NA1_5551012965,BjpGrpECtM-xX91hBqd3tuHZ0iwnWC7BKgQN_KwvKO1u9M...,goongi,000,100,0,0,0,804,Yunara,...,0,0,0,0.0,0,0,-0.285004,0.702374,0,0.000000
4,NA1_5551012965,NWtFU9iWQ5vlHW5dVHEmTdabdfDEZT1tT__b47esTV8hFT...,Volg Zangief,ALX,100,0,0,0,12,Alistar,...,0,0,0,0.0,0,0,0.175785,3.363126,3,0.673394
